

# AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  


---




In [1]:
import pandas as pd        # For working with data tables (like Excel in Python)
import numpy as np         # For math and generating random numbers

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [2]:
np.random.seed(42)

print('Random seed set to 42.')
print('   This means our data will be identical every time we run the notebook.')

Random seed set to 42.
   This means our data will be identical every time we run the notebook.


In [3]:
customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']

countries = [
    'USA', 'UK', 'India', 'Cayman Islands', 'Panama',
    'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore'
]

# High-risk countries flagged by FATF
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']

transaction_types = [
    'Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
    'Online Transfer', 'Check', 'Crypto Exchange'
]

print('Categories defined.')
print(f'   Total countries     : {len(countries)}')
print(f'   High-risk countries : {high_risk_countries}')
print(f'   Transaction types   : {transaction_types}')

Categories defined.
   Total countries     : 10
   High-risk countries : ['Cayman Islands', 'Panama', 'Nigeria']
   Transaction types   : ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal', 'Online Transfer', 'Check', 'Crypto Exchange']


In [4]:
n = 500  # Number of transactions

df = pd.DataFrame({

    # Unique ID for each transaction
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],

    # Customer ID — some customers repeat (realistic)
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],

    # Type of customer
    'Customer_Type': np.random.choice(
        customer_types, n,
        p=[0.5, 0.3, 0.1, 0.1]  # 50% Individual, 30% Business, 10% Shell, 10% NGO
    ),

    # Transaction Amount (with 5% structuring pattern built in)
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),                          # Suspicious
            np.random.exponential(scale=3000, size=n).clip(100, 100000) # Normal
        ), 2),

    # Type of transaction
    'Transaction_Type': np.random.choice(transaction_types, n),

    # Where the money is coming FROM
    'Origin_Country': np.random.choice(
        countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]
    ),

    # Where the money is going TO
    'Destination_Country': np.random.choice(countries, n),

    # How many times this customer transacted in the last 30 days
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),

    # This customer's average transaction over the last 6 months
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2
    ),

    # How old is this account (in years)
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),

    # Has a Suspicious Activity Report (SAR) been filed for this customer before?
    # SAR = report filed by banks to regulators (FinCEN in USA, FIU-IND in India)
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

print(f'Dataset created with {len(df)} transactions and {len(df.columns)} columns!')

Dataset created with 500 transactions and 11 columns!


In [5]:
print('First 5 rows of your dataset:')
df.head()

First 5 rows of your dataset:


,Transaction_ID,Customer_ID,Customer_Type,Transaction_Amount,Transaction_Type,Origin_Country,Destination_Country,Num_Transactions_Last_30Days,Avg_Transaction_Last_6Months,Account_Age_Years,Prior_SAR_Filed
0,TXN00001,CUST1102,NGO,1392.82,Online Transfer,USA,USA,30,141.81,12.7,0
1,TXN00002,CUST1435,Individual,927.13,Crypto Exchange,Panama,USA,46,1081.43,15.7,1
2,TXN00003,CUST1860,Individual,9325.60,Online Transfer,USA,Cayman Islands,20,247.41,12.0,0
3,TXN00004,CUST1270,Business,2075.45,Crypto Exchange,Switzerland,Germany,39,1897.61,8.3,0
4,TXN00005,CUST1106,Individual,927.82,Wire Transfer,Germany,USA,12,5576.86,19.2,0


In [6]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
for col in df.columns:
    print(f'  - {col}')

Dataset shape: (500, 11)

Column names:
  - Transaction_ID
  - Customer_ID
  - Customer_Type
  - Transaction_Amount
  - Transaction_Type
  - Origin_Country
  - Destination_Country
  - Num_Transactions_Last_30Days
  - Avg_Transaction_Last_6Months
  - Account_Age_Years
  - Prior_SAR_Filed


In [7]:
print('Transaction Amount Statistics:')
df['Transaction_Amount'].describe().round(2)

Transaction Amount Statistics:


,Transaction_Amount
count,500.00
mean,3451.82
std,3355.96
min,100.00
25%,830.46
50%,2358.64
75%,4878.56
max,17506.62


In [8]:
print('Customer Type Distribution:')
df['Customer_Type'].value_counts()

Customer Type Distribution:


,count
Customer_Type,
Individual,249
Business,154
Shell Company,49
NGO,48


In [9]:
print('Top Origin Countries:')
df['Origin_Country'].value_counts()

Top Origin Countries:


,count
Origin_Country,
USA,156
UK,76
India,63
Switzerland,50
Germany,48
Nigeria,28
Panama,25
Cayman Islands,21
UAE,18
